# GHIA — Ferramental Quantitativo | Fase 0: Infraestrutura de Coleta de Dados

Protótipo de coleta e tratamento das séries macro e de mercado que servem de base para as fases
seguintes do projeto (faixa de erro histórico do Focus, distribuição condicional de retorno por
classe de ativo, Black-Litterman).

**Fontes:** SGS/Banco Central, SIDRA/IBGE, Boletim Focus (BCB) e IPEADATA.

**Saída:** base mensal única em Parquet, com log de execução e checagem de consistência.

In [1]:
from bcb import sgs
import sidrapy
import ipeadatapy as ipea
import requests
import pandas as pd
import numpy as np
import pyarrow  # engine de parquet — se faltar, instale com `pip install pyarrow` antes de continuar
from datetime import datetime
import json
import logging
import time
import inspect
from pathlib import Path

pd.set_option('display.width', 120)

## 0. Configuração geral (pastas, log, parâmetros)

In [2]:
# Pastas de saída. DATA_DIR guarda a base tratada; RAW_DIR guarda o dado bruto de cada fonte
# (útil para auditar uma coleta que deu errado sem precisar buscar tudo de novo).

DATA_DIR = Path("dados_ghia")
RAW_DIR = DATA_DIR / "raw"
DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

LOG_PATH = DATA_DIR / "log_atualizacao.jsonl"
# 2003-01-01: teto natural do painel. As 7 séries do SGS coletadas aqui têm histórico
# desde 1980-1988, exceto o IBC-Br (código 24364), que só existe a partir daí — pedir
# uma data anterior não traria mais dado, só mais NaN. Fora do SGS, vale reparar que a
# taxa de desocupação (SIDRA/PNAD Contínua) só começa em ~12/2012, então qualquer
# modelo que a use como variável vai ter uma janela comum bem mais curta — ver a
# checagem de cobertura por coluna na Seção 6.
DATA_INICIO = "2003-01-01"

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("ghia_coleta")


def registrar_log(etapa: str, status: str, detalhes: dict | None = None) -> None:
    """
    Registra uma linha de log estruturado (JSON Lines) com o resultado de uma etapa de coleta.

    `status` deve ser "ok" ou "erro". O arquivo acumula um histórico de execuções, permitindo
    checar depois quando cada fonte foi atualizada com sucesso pela última vez.
    """
    registro = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "etapa": etapa,
        "status": status,
        "detalhes": detalhes or {},
    }
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")
    nivel = logging.INFO if status == "ok" else logging.WARNING
    logger.log(nivel, f"[{etapa}] {status} | {detalhes or ''}")


def salvar_parquet_seguro(df: pd.DataFrame, caminho: Path) -> None:
    """
    Salva um DataFrame em parquet sem derrubar a coleta se isso falhar. O parquet bruto
    (pasta `raw/`) é só um artefato de auditoria — a ausência dele não pode custar os
    dados que já foram coletados com sucesso, por isso o erro aqui vira só um aviso no
    log em vez de propagar e ser pego pelo try/except da coleta (que descartaria o
    DataFrame inteiro por causa de uma falha na gravação, não na coleta em si).
    """
    try:
        df.to_parquet(caminho)
    except Exception as e:
        registrar_log("salvar_parquet_raw", "erro", {"caminho": str(caminho), "mensagem": str(e)})


def executar_com_retentativas(func, *args, tentativas: int = 4, espera_inicial: int = 2, **kwargs):
    """
    Executa `func(*args, **kwargs)` com retentativas e backoff exponencial (2s, 4s, 8s,
    16s...). Usado em toda chamada de rede da coleta: uma falha transitória (timeout,
    conexão instável, rede corporativa mais lenta) não deve derrubar a coleta inteira
    na primeira tentativa.
    """
    for tentativa in range(tentativas):
        try:
            return func(*args, **kwargs)
        except Exception:
            if tentativa == tentativas - 1:
                raise
            time.sleep(espera_inicial * (2 ** tentativa))

## 1. Coleta — SGS (Banco Central)

Séries diárias e mensais do Sistema Gerenciador de Séries Temporais. Códigos conferidos
diretamente na API (`api.bcb.gov.br`) em 2026-08-29.

In [3]:
# (código, é_diária) — série mensal em toda a base, de propósito: mais simples de
# tratar (sem precisar mensualizar nem respeitar o limite de 10 anos por consulta que
# o SGS impõe a séries diárias) e suficiente para o cenário macro que a Fase 2 usa.
# O sinalizador "é_diária" continua existindo caso algum dia seja preciso adicionar uma
# série diária de novo (ex.: para um estudo de alta frequência).
CODIGOS_SGS = {
    "selic_over_mensal": (4390, False),     # Selic acumulada no mês, anualizada (% a.a.) — usada como nível de juros
    "ipca_mensal": (433, False),            # IPCA - variação mensal (%)
    "ipca_12m": (13522, False),             # IPCA - variação acumulada em 12 meses (%)
    "cambio_fim_mes": (3695, False),        # Dólar americano (compra), câmbio livre, fim de período mensal
    "ibcbr_dessaz": (24364, False),         # IBC-Br, com ajuste sazonal
    "credito_saldo_total": (20539, False),  # Saldo da carteira de crédito - total (R$ milhões)
    "cdi_mensal": (4391, False),            # CDI acumulado no mês (%) — usado como retorno da classe CDI na Fase 2
}


def _dividir_em_janelas(inicio: pd.Timestamp, fim: pd.Timestamp, anos: int = 9) -> list[tuple]:
    """Quebra um intervalo de datas em janelas de até `anos` anos (fecho à direita inclusive)."""
    janelas = []
    cursor = inicio
    while cursor < fim:
        fim_janela = min(cursor + pd.DateOffset(years=anos, days=-1), fim)
        janelas.append((cursor, fim_janela))
        cursor = fim_janela + pd.DateOffset(days=1)
    return janelas


SGS_SUPORTA_TIMEOUT = "timeout" in inspect.signature(sgs.get).parameters


def _sgs_get(codigos: dict, start: str, end: str) -> pd.DataFrame:
    """
    Chama `sgs.get`, passando `timeout` só se a versão instalada do python-bcb suportar
    o parâmetro — versões mais antigas da biblioteca não têm esse argumento e quebram
    com TypeError se ele for passado.
    """
    if SGS_SUPORTA_TIMEOUT:
        return sgs.get(codigos, start=start, end=end, timeout=60)
    return sgs.get(codigos, start=start, end=end)


def _coletar_serie_sgs(nome: str, codigo: int, data_inicio: str, diaria: bool) -> pd.DataFrame:
    """
    Coleta uma única série do SGS. Uma série por chamada (em vez de várias num só
    `sgs.get`) — testado e mais estável que pedir várias séries de uma vez, que se
    mostrou instável nesta rede. Séries diárias são buscadas em janelas de até 9 anos
    para respeitar o limite de 10 anos da API. Timeout generoso (60s, quando suportado)
    e retentativas: numa conexão mais lenta (rede corporativa, VPN) o timeout padrão da
    biblioteca estoura antes de a API do BCB responder.
    """
    inicio, fim = pd.Timestamp(data_inicio), pd.Timestamp.today()
    janelas = _dividir_em_janelas(inicio, fim) if diaria else [(inicio, fim)]
    partes = [
        executar_com_retentativas(_sgs_get, {nome: codigo}, str(i.date()), str(f.date()))
        for i, f in janelas
    ]
    serie = pd.concat(partes).sort_index()
    return serie[~serie.index.duplicated(keep="last")]


def coletar_sgs(codigos: dict, data_inicio: str = DATA_INICIO) -> pd.DataFrame:
    """
    Coleta múltiplas séries do SGS/Banco Central e retorna um único DataFrame indexado
    por data (uma coluna por série). Séries de frequências diferentes (diária, mensal)
    ficam com NaN nas datas em que não há observação — o alinhamento de calendário é
    responsabilidade da etapa de tratamento, não da coleta.
    """
    colunas = [_coletar_serie_sgs(nome, codigo, data_inicio, diaria) for nome, (codigo, diaria) in codigos.items()]
    df = pd.concat(colunas, axis=1)
    df.index.name = "data"
    return df


try:
    df_sgs = coletar_sgs(CODIGOS_SGS)
    salvar_parquet_seguro(df_sgs, RAW_DIR / "sgs.parquet")
    registrar_log("coleta_sgs", "ok", {"n_series": len(CODIGOS_SGS), "n_obs": len(df_sgs)})
except Exception as e:
    df_sgs = pd.DataFrame()
    registrar_log("coleta_sgs", "erro", {"mensagem": str(e)})

df_sgs.tail()

2026-09-18 03:52:45,441 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.4390/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=18%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-18 03:52:46,536 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=18%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-18 03:52:47,907 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.13522/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=18%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-18 03:52:49,075 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.3695/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=18%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-18 03:52:50,288 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.24364/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=18%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-18 03:52:51,357 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.20539/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=18%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-18 03:52:52,535 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.4391/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=18%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-18 03:52:52,578 | INFO | [coleta_sgs] ok | {'n_series': 7, 'n_obs': 285}


,selic_over_mensal,ipca_mensal,ipca_12m,cambio_fim_mes,ibcbr_dessaz,credito_saldo_total,cdi_mensal
data,,,,,,,
2026-05-01,1.07,0.58,4.72,5.0563,111.13846,7305306.0,1.07
2026-06-01,1.12,0.16,4.64,5.1760,110.15775,7353293.0,1.12
2026-07-01,1.22,0.07,4.44,5.0767,109.91611,7372243.0,1.22
2026-08-01,1.09,-0.32,4.22,5.1810,NaN,NaN,1.09
2026-09-01,0.62,NaN,NaN,NaN,NaN,NaN,0.57


## 2. Coleta — SIDRA (IBGE)

Taxa de desocupação da PNAD Contínua (trimestre móvel), tabela 6381, agregado Brasil.

In [4]:
def coletar_sidra_desocupacao() -> pd.DataFrame:
    """
    Coleta a série completa da taxa de desocupação (PNAD Contínua, trimestre móvel,
    Brasil) via SIDRA e retorna um DataFrame mensal com a taxa em pontos percentuais.
    A data atribuída a cada trimestre móvel é o último mês do trimestre.
    """
    bruto = executar_com_retentativas(
        sidrapy.get_table,
        table_code="6381",
        territorial_level="1",
        ibge_territorial_code="all",
        variable="4099",
        period="all",
    )
    bruto = bruto.iloc[1:].copy()  # primeira linha é o cabeçalho descritivo (D2C = Trimestre Móvel)
    bruto["data"] = pd.to_datetime(bruto["D2C"], format="%Y%m")
    bruto["taxa_desocupacao"] = pd.to_numeric(bruto["V"], errors="coerce")
    return bruto.set_index("data")[["taxa_desocupacao"]].sort_index()


try:
    df_sidra = coletar_sidra_desocupacao()
    salvar_parquet_seguro(df_sidra, RAW_DIR / "sidra_desocupacao.parquet")
    registrar_log("coleta_sidra", "ok", {"n_obs": len(df_sidra)})
except Exception as e:
    df_sidra = pd.DataFrame()
    registrar_log("coleta_sidra", "erro", {"mensagem": str(e)})

df_sidra.tail()

2026-09-18 03:53:08,502 | WARNING | [coleta_sidra] erro | {'mensagem': '<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><meta http-equiv="content-security-policy" content="default-src &#39;none&#39;; script-src &#39;nonce-cosa9G3FCfWuTFgUTwJ6M7&#39; &#39;unsafe-eval&#39; https://challenges.cloudflare.com; script-src-attr &#39;none&#39;; style-src &#39;unsafe-inline&#39;; img-src &#39;self&#39; https://challenges.cloudflare.com; connect-src &#39;self&#39; https://challenges.cloudflare.com; frame-src &#39;self&#39; https://challenges.cloudflare.com blob:; child-src &#39;self&#39; https://challenges.cloudflare.com blob:; worker-src blob:; form-action http: https:; base-uri &#39;self&#39;"><style>*{box-sizing:border-box;margin:0;padding:

""


## 3. Coleta — Boletim Focus (expectativas de mercado, BCB)

Serviço OData `Expectativas` do BCB (Olinda). Duas séries relevantes para a Fase 1
(faixa de erro histórico do Focus):

- `ExpectativaMercadoMensais`: mediana/média por indicador e mês de referência.
- `ExpectativasMercadoInflacao12Meses`: mediana/média da inflação acumulada nos
  próximos 12 meses — é a série de horizonte fixo que permite medir erro por horizonte
  sem ter que reconstruir o "horizonte" a partir da data de referência.

In [5]:
FOCUS_BASE_URL = "https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata"


def _consultar_focus_odata(entidade: str, filtro: str, orderby: str = "Data") -> pd.DataFrame:
    """
    Faz uma consulta genérica ao serviço OData de Expectativas do BCB, paginando em
    blocos de 1000 registros (limite do serviço) até esgotar o resultado.
    """
    registros = []
    skip = 0
    tamanho_pagina = 1000
    while True:
        url = (
            f"{FOCUS_BASE_URL}/{entidade}"
            f"?$filter={filtro}&$orderby={orderby}&$format=json"
            f"&$top={tamanho_pagina}&$skip={skip}"
        )
        resp = executar_com_retentativas(requests.get, url, timeout=60)
        resp.raise_for_status()
        pagina = resp.json()["value"]
        registros.extend(pagina)
        if len(pagina) < tamanho_pagina:
            break
        skip += tamanho_pagina
    return pd.DataFrame(registros)


def coletar_focus_ipca_12m(suavizada: str = "N") -> pd.DataFrame:
    """
    Coleta a série de expectativa de IPCA acumulado em 12 meses à frente (horizonte fixo),
    versão não suavizada por padrão. Retorna média, mediana e desvio-padrão por data de
    coleta, indexado por data.
    """
    # baseCalculo=0 ("todas as coletas do dia") garante uma linha por data; baseCalculo=1
    # é uma variante intradiária (coletas antes do horário de corte) que duplicaria o índice.
    filtro = f"Indicador eq 'IPCA' and Suavizada eq '{suavizada}' and baseCalculo eq 0"
    df = _consultar_focus_odata("ExpectativasMercadoInflacao12Meses", filtro)
    df["Data"] = pd.to_datetime(df["Data"])
    return df.set_index("Data").sort_index()


def coletar_focus_mensal(indicador: str = "IPCA", anos_recentes: int = 2) -> pd.DataFrame:
    """
    Coleta a série de expectativas mensais (por mês de referência) para um indicador do
    Focus (ex.: IPCA, Câmbio, Selic). Cada linha é uma combinação (data de coleta, mês de
    referência) — granularidade fina, útil para horizontes diferentes de 12 meses.

    Limitada aos últimos `anos_recentes` anos por desempenho: essa tabela tem frequência
    diária cruzada com dezenas de meses de referência, e o serviço do BCB pagina devagar
    em consultas longas (~1.5s por 1000 linhas). Para o histórico completo em produção,
    rodar esta função como job em lote separado, fora deste notebook interativo.
    """
    data_minima = (pd.Timestamp.today() - pd.DateOffset(years=anos_recentes)).strftime("%Y-%m-%d")
    filtro = f"Indicador eq '{indicador}' and Data ge '{data_minima}'"
    df = _consultar_focus_odata("ExpectativaMercadoMensais", filtro)
    df["Data"] = pd.to_datetime(df["Data"])
    df["DataReferencia"] = pd.to_datetime(df["DataReferencia"], format="%m/%Y")
    return df.sort_values(["DataReferencia", "Data"])


try:
    df_focus_12m = coletar_focus_ipca_12m()
    salvar_parquet_seguro(df_focus_12m, RAW_DIR / "focus_ipca_12m.parquet")
    registrar_log("coleta_focus_12m", "ok", {"n_obs": len(df_focus_12m)})
except Exception as e:
    df_focus_12m = pd.DataFrame()
    registrar_log("coleta_focus_12m", "erro", {"mensagem": str(e)})

try:
    df_focus_ipca_mensal = coletar_focus_mensal("IPCA")
    salvar_parquet_seguro(df_focus_ipca_mensal, RAW_DIR / "focus_ipca_mensal.parquet")
    registrar_log("coleta_focus_mensal", "ok", {"n_obs": len(df_focus_ipca_mensal)})
except Exception as e:
    df_focus_ipca_mensal = pd.DataFrame()
    registrar_log("coleta_focus_mensal", "erro", {"mensagem": str(e)})

df_focus_12m.tail()

2026-09-18 03:53:20,817 | INFO | [coleta_focus_12m] ok | {'n_obs': 6236}


2026-09-18 03:54:03,123 | INFO | [coleta_focus_mensal] ok | {'n_obs': 24900}


,Indicador,Suavizada,Media,Mediana,DesvioPadrao,Minimo,Maximo,numeroRespondentes,baseCalculo
Data,,,,,,,,,
2026-09-04,IPCA,N,4.3023,4.3110,0.5113,2.6808,5.9637,133.0,0
2026-09-08,IPCA,N,4.3028,4.3151,0.5103,2.6808,5.9637,132.0,0
2026-09-09,IPCA,N,4.3024,4.3055,0.5087,2.6808,5.9637,132.0,0
2026-09-10,IPCA,N,4.3020,4.3110,0.5063,2.6808,5.9637,133.0,0
2026-09-11,IPCA,N,4.6992,4.6548,0.5056,3.1679,6.1942,135.0,0


## 4. Coleta — IPEADATA

Fonte adicional mencionada no briefing. A API pública do IPEADATA é historicamente
instável (fora do ar com frequência) — por isso a coleta é isolada em try/except e não
derruba o pipeline caso falhe; a base tratada final simplesmente sai sem essas colunas
naquela execução, e o log registra o problema para investigar depois.

In [6]:
CODIGOS_IPEADATA = {
    "ipca_ipea_12m": "PRECOS12_IPCAGA12",      # IPCA, var. acumulada 12 meses (%) — cruzamento com SGS
    "expectativa_ipca_ipea": "BM12_IPCAEXP1212",  # Expectativa média de inflação 12 meses (%)
}


def coletar_ipeadata(codigos: dict) -> pd.DataFrame:
    """
    Coleta múltiplas séries do IPEADATA e as consolida em um único DataFrame mensal.
    Mantém apenas a última coluna de cada série retornada pela biblioteca (valor numérico),
    descartando metadados redundantes.
    """
    colunas = []
    for nome, codigo in codigos.items():
        serie = ipea.timeseries(codigo).iloc[:, [-1]]
        serie.columns = [nome]
        colunas.append(serie)
    return pd.concat(colunas, axis=1)


try:
    df_ipea = coletar_ipeadata(CODIGOS_IPEADATA)
    salvar_parquet_seguro(df_ipea, RAW_DIR / "ipeadata.parquet")
    registrar_log("coleta_ipeadata", "ok", {"n_series": len(CODIGOS_IPEADATA), "n_obs": len(df_ipea)})
except Exception as e:
    df_ipea = pd.DataFrame()
    registrar_log("coleta_ipeadata", "erro", {"mensagem": str(e)})

df_ipea.tail()

2026-09-18 03:54:11,566 | INFO | [coleta_ipeadata] ok | {'n_series': 2, 'n_obs': 549}


,ipca_ipea_12m,expectativa_ipca_ipea
DATE,,
2026-04-01,4.39,4.2360
2026-05-01,4.72,4.1949
2026-06-01,4.64,4.2177
2026-07-01,4.44,4.0830
2026-08-01,4.22,4.2931


## 5. Tratamento — consolidação em base mensal única

Toda série do SGS já é coletada mensal (Seção 1), então esta etapa é, na prática, só
reindexar tudo para o primeiro dia do mês e juntar as fontes. A função `mensualizar`
continua aceitando uma lista de colunas diárias para tirar a média do mês — hoje vazia,
mas pronta caso algum dia entre uma série diária de novo (ex.: estudo de alta
frequência). Nenhuma dessazonalização é aplicada aqui — o IBC-Br e a taxa de
desocupação já vêm dessazonalizados na fonte; séries que precisarem de ajuste
específico por classe de ativo entram nessa etapa nas fases seguintes, não na
infraestrutura de coleta.

In [7]:
SERIES_DIARIAS = [nome for nome, (_, diaria) in CODIGOS_SGS.items() if diaria]


def mensualizar(df: pd.DataFrame, colunas_diarias: list[str]) -> pd.DataFrame:
    """
    Recebe um DataFrame com índice de data (frequência mista) e devolve uma versão
    mensal: colunas em `colunas_diarias` são agregadas pela média do mês; as demais
    colunas são reamostradas por último valor não nulo do mês (já são mensais na origem).
    """
    diarias = df[colunas_diarias].resample("MS").mean()
    mensais = df.drop(columns=colunas_diarias).resample("MS").last()
    return diarias.join(mensais, how="outer")


df_sgs_mensal = mensualizar(df_sgs, SERIES_DIARIAS) if not df_sgs.empty else pd.DataFrame()
df_sidra_mensal = df_sidra.resample("MS").last() if not df_sidra.empty else pd.DataFrame()

base = df_sgs_mensal.join(df_sidra_mensal, how="outer")
if not df_ipea.empty:
    df_ipea.index = pd.to_datetime(df_ipea.index)
    base = base.join(df_ipea.resample("MS").last(), how="outer")

# IPEADATA tem histórico bem mais longo que as outras fontes (ex.: séries desde 1980) —
# sem este corte, o join externo estenderia "base" para muito antes de DATA_INICIO só
# por causa de uma fonte, contradizendo a data de início pedida para todas as outras.
base = base.sort_index().loc[DATA_INICIO:]
base.tail()

,selic_over_mensal,ipca_mensal,ipca_12m,cambio_fim_mes,ibcbr_dessaz,credito_saldo_total,cdi_mensal,ipca_ipea_12m,expectativa_ipca_ipea
data,,,,,,,,,
2026-05-01,1.07,0.58,4.72,5.0563,111.13846,7305306.0,1.07,4.72,4.1949
2026-06-01,1.12,0.16,4.64,5.1760,110.15775,7353293.0,1.12,4.64,4.2177
2026-07-01,1.22,0.07,4.44,5.0767,109.91611,7372243.0,1.22,4.44,4.0830
2026-08-01,1.09,-0.32,4.22,5.1810,NaN,NaN,1.09,4.22,4.2931
2026-09-01,0.62,NaN,NaN,NaN,NaN,NaN,0.57,NaN,NaN


## 6. Checagem de consistência

In [8]:
def checar_consistencia(df: pd.DataFrame) -> dict:
    """
    Roda checagens básicas de qualidade sobre a base consolidada: percentual de nulos
    por coluna, existência de datas duplicadas e maior gap (em meses) sem nenhuma
    observação registrada em pelo menos uma série. Não corrige nada — só relata.

    Se `df` vier vazio (nenhuma linha), é sinal de que uma das coletas anteriores
    falhou silenciosamente — os try/except de cada fonte evitam que o pipeline pare,
    mas isso pode mascarar o problema até aqui. Nesse caso, a função não tenta calcular
    período/gaps (não há o que calcular) e devolve um relatório sinalizando a falha, em
    vez de estourar um erro genérico do pandas ao comparar datas com NaT.
    """
    if df.empty:
        return {
            "erro": "base vazia — verifique os logs de coleta acima (SGS, SIDRA, Focus, IPEADATA) para achar qual fonte falhou",
            "n_linhas": 0,
        }

    duplicadas = int(df.index.duplicated().sum())
    nulos_pct = (df.isna().mean() * 100).round(1).to_dict()

    calendario_completo = pd.date_range(df.index.min(), df.index.max(), freq="MS")
    meses_faltantes = calendario_completo.difference(df.index)

    return {
        "periodo": [str(df.index.min().date()), str(df.index.max().date())],
        "n_linhas": len(df),
        "datas_duplicadas": duplicadas,
        "nulos_pct_por_coluna": nulos_pct,
        "meses_faltantes_no_indice": [str(d.date()) for d in meses_faltantes],
    }


relatorio = checar_consistencia(base)
registrar_log("checagem_consistencia", "ok", relatorio)
relatorio

2026-09-18 03:54:11,609 | INFO | [checagem_consistencia] ok | {'periodo': ['2003-01-01', '2026-09-01'], 'n_linhas': 285, 'datas_duplicadas': 0, 'nulos_pct_por_coluna': {'selic_over_mensal': 0.0, 'ipca_mensal': 0.4, 'ipca_12m': 0.4, 'cambio_fim_mes': 0.4, 'ibcbr_dessaz': 0.7, 'credito_saldo_total': 0.7, 'cdi_mensal': 0.0, 'ipca_ipea_12m': 0.4, 'expectativa_ipca_ipea': 0.4}, 'meses_faltantes_no_indice': []}


{'periodo': ['2003-01-01', '2026-09-01'],
 'n_linhas': 285,
 'datas_duplicadas': 0,
 'nulos_pct_por_coluna': {'selic_over_mensal': 0.0,
  'ipca_mensal': 0.4,
  'ipca_12m': 0.4,
  'cambio_fim_mes': 0.4,
  'ibcbr_dessaz': 0.7,
  'credito_saldo_total': 0.7,
  'cdi_mensal': 0.0,
  'ipca_ipea_12m': 0.4,
  'expectativa_ipca_ipea': 0.4},
 'meses_faltantes_no_indice': []}

### 6.1 Cobertura temporal por coluna

O período no relatório acima é o do índice como um todo (`base.index.min()` a
`base.index.max()`) — não diz onde cada série *começa de fato*. Essencial antes de
rodar qualquer modelo: se as colunas não cobrem o mesmo intervalo, alinhar "na unha"
com um `dropna()` genérico corta a amostra sem deixar claro qual coluna mandou no
corte. A tabela abaixo mostra a primeira e a última data válida de cada série.

In [9]:
def relatorio_cobertura_temporal(df: pd.DataFrame) -> pd.DataFrame:
    """
    Para cada coluna, reporta a primeira e a última data com valor não nulo e o total
    de observações válidas. Junto com `checar_consistencia`, deixa explícito — em vez
    de implícito dentro de um dropna() — de onde até onde cada série realmente cobre,
    para que o corte usado em qualquer modelo seja uma decisão visível, não um efeito
    colateral silencioso do alinhamento.
    """
    linhas = []
    for coluna in df.columns:
        valida = df[coluna].dropna()
        linhas.append({
            "coluna": coluna,
            "primeira_data_valida": valida.index.min().date() if not valida.empty else None,
            "ultima_data_valida": valida.index.max().date() if not valida.empty else None,
            "n_obs_validas": len(valida),
        })
    return pd.DataFrame(linhas).set_index("coluna")


cobertura_temporal = relatorio_cobertura_temporal(base)
registrar_log("cobertura_temporal", "ok", cobertura_temporal.astype(str).to_dict(orient="index"))
cobertura_temporal

2026-09-18 03:54:11,624 | INFO | [cobertura_temporal] ok | {'selic_over_mensal': {'primeira_data_valida': '2003-01-01', 'ultima_data_valida': '2026-09-01', 'n_obs_validas': '285'}, 'ipca_mensal': {'primeira_data_valida': '2003-01-01', 'ultima_data_valida': '2026-08-01', 'n_obs_validas': '284'}, 'ipca_12m': {'primeira_data_valida': '2003-01-01', 'ultima_data_valida': '2026-08-01', 'n_obs_validas': '284'}, 'cambio_fim_mes': {'primeira_data_valida': '2003-01-01', 'ultima_data_valida': '2026-08-01', 'n_obs_validas': '284'}, 'ibcbr_dessaz': {'primeira_data_valida': '2003-01-01', 'ultima_data_valida': '2026-07-01', 'n_obs_validas': '283'}, 'credito_saldo_total': {'primeira_data_valida': '2003-01-01', 'ultima_data_valida': '2026-07-01', 'n_obs_validas': '283'}, 'cdi_mensal': {'primeira_data_valida': '2003-01-01', 'ultima_data_valida': '2026-09-01', 'n_obs_validas': '285'}, 'ipca_ipea_12m': {'primeira_data_valida': '2003-01-01', 'ultima_data_valida': '2026-08-01', 'n_obs_validas': '284'}, 'exp

,primeira_data_valida,ultima_data_valida,n_obs_validas
coluna,,,
selic_over_mensal,2003-01-01,2026-09-01,285
ipca_mensal,2003-01-01,2026-08-01,284
ipca_12m,2003-01-01,2026-08-01,284
cambio_fim_mes,2003-01-01,2026-08-01,284
ibcbr_dessaz,2003-01-01,2026-07-01,283
credito_saldo_total,2003-01-01,2026-07-01,283
cdi_mensal,2003-01-01,2026-09-01,285
ipca_ipea_12m,2003-01-01,2026-08-01,284
expectativa_ipca_ipea,2003-01-01,2026-08-01,284


### 6.2 Base de modelagem: painel alinhado a partir de 03/2012

`base` continua com o histórico cheio desde 2003 (bom para auditoria e para qualquer
análise que não precise da taxa de desocupação). Mas a taxa de desocupação (SIDRA) só
existe a partir de 03/2012 — é o teto real de um painel com TODAS as colunas presentes
ao mesmo tempo. Para qualquer modelo (a Fase 2 usa isso a partir daqui), o corte é
explícito nesta célula, não um efeito colateral escondido dentro de um `dropna()`
mais adiante: mostra quantas linhas saem, de onde até onde, e confirma se ainda sobra
algum buraco dentro da janela escolhida.

In [10]:
DATA_INICIO_MODELAGEM = "2012-03-01"  # primeira data com taxa de desocupação (SIDRA) disponível

n_antes = len(base)
base_modelagem = base.loc[DATA_INICIO_MODELAGEM:].copy()
n_depois = len(base_modelagem)

print(f"Base de modelagem: corte em {DATA_INICIO_MODELAGEM} -> {n_antes} para {n_depois} linhas "
      f"({n_antes - n_depois} descartadas, todas anteriores a essa data).")

nulos_restantes = base_modelagem.isna().sum()
nulos_restantes = nulos_restantes[nulos_restantes > 0]
if len(nulos_restantes) > 0:
    print("Ainda há valores faltantes DENTRO dessa janela (não descartados aqui — cada modelo decide o que fazer com eles):")
    print(nulos_restantes)
else:
    print("Nenhum valor faltante dentro da janela de modelagem: painel totalmente alinhado.")

registrar_log("base_modelagem", "ok", {
    "data_inicio": DATA_INICIO_MODELAGEM, "n_linhas": n_depois,
    "linhas_descartadas": n_antes - n_depois, "nulos_restantes": nulos_restantes.to_dict(),
})
base_modelagem.tail()

2026-09-18 03:54:11,637 | INFO | [base_modelagem] ok | {'data_inicio': '2012-03-01', 'n_linhas': 175, 'linhas_descartadas': 110, 'nulos_restantes': {'ipca_mensal': 1, 'ipca_12m': 1, 'cambio_fim_mes': 1, 'ibcbr_dessaz': 2, 'credito_saldo_total': 2, 'ipca_ipea_12m': 1, 'expectativa_ipca_ipea': 1}}


Base de modelagem: corte em 2012-03-01 -> 285 para 175 linhas (110 descartadas, todas anteriores a essa data).
Ainda há valores faltantes DENTRO dessa janela (não descartados aqui — cada modelo decide o que fazer com eles):
ipca_mensal              1
ipca_12m                 1
cambio_fim_mes           1
ibcbr_dessaz             2
credito_saldo_total      2
ipca_ipea_12m            1
expectativa_ipca_ipea    1
dtype: int64


,selic_over_mensal,ipca_mensal,ipca_12m,cambio_fim_mes,ibcbr_dessaz,credito_saldo_total,cdi_mensal,ipca_ipea_12m,expectativa_ipca_ipea
data,,,,,,,,,
2026-05-01,1.07,0.58,4.72,5.0563,111.13846,7305306.0,1.07,4.72,4.1949
2026-06-01,1.12,0.16,4.64,5.1760,110.15775,7353293.0,1.12,4.64,4.2177
2026-07-01,1.22,0.07,4.44,5.0767,109.91611,7372243.0,1.22,4.44,4.0830
2026-08-01,1.09,-0.32,4.22,5.1810,NaN,NaN,1.09,4.22,4.2931
2026-09-01,0.62,NaN,NaN,NaN,NaN,NaN,0.57,NaN,NaN


## 7. Exportação da base tratada

In [11]:
# A base tratada é a entrega real da Fase 0 — ao contrário do parquet bruto (só
# auditoria), aqui uma falha não pode passar batido. Mas se o ambiente tiver um
# pyarrow quebrado/incompatível (mesmo problema que geraria o aviso na Seção 1), cai
# para CSV em vez de travar o notebook inteiro por causa de um formato de arquivo.
# Exporta as duas versões: histórico completo (auditoria) e o painel alinhado a partir
# de 03/2012 (o que qualquer modelo deveria consumir).
CAMINHO_BASE = DATA_DIR / "base_ghia_mensal.parquet"
CAMINHO_BASE_MODELAGEM = DATA_DIR / "base_ghia_modelagem.parquet"
try:
    base.to_parquet(CAMINHO_BASE)
    base_modelagem.to_parquet(CAMINHO_BASE_MODELAGEM)
    registrar_log("exportacao_base", "ok", {"caminho": str(CAMINHO_BASE), "n_linhas": len(base), "n_colunas": base.shape[1]})
except Exception as e:
    CAMINHO_BASE = DATA_DIR / "base_ghia_mensal.csv"
    CAMINHO_BASE_MODELAGEM = DATA_DIR / "base_ghia_modelagem.csv"
    base.to_csv(CAMINHO_BASE)
    base_modelagem.to_csv(CAMINHO_BASE_MODELAGEM)
    registrar_log("exportacao_base", "fallback_csv", {"caminho": str(CAMINHO_BASE), "mensagem_parquet": str(e)})
    print(f"Aviso: parquet falhou ({e}). Exportado em CSV: instale/atualize o pyarrow (`pip install -U pyarrow`) para voltar a usar parquet.")

print(f"Base completa exportada: {CAMINHO_BASE} ({base.shape[0]} linhas x {base.shape[1]} colunas)")
print(f"Base de modelagem exportada: {CAMINHO_BASE_MODELAGEM} ({base_modelagem.shape[0]} linhas x {base_modelagem.shape[1]} colunas)")
base.tail()

2026-09-18 03:54:11,656 | INFO | [exportacao_base] ok | {'caminho': 'dados_ghia/base_ghia_mensal.parquet', 'n_linhas': 285, 'n_colunas': 9}


Base completa exportada: dados_ghia/base_ghia_mensal.parquet (285 linhas x 9 colunas)
Base de modelagem exportada: dados_ghia/base_ghia_modelagem.parquet (175 linhas x 9 colunas)


,selic_over_mensal,ipca_mensal,ipca_12m,cambio_fim_mes,ibcbr_dessaz,credito_saldo_total,cdi_mensal,ipca_ipea_12m,expectativa_ipca_ipea
data,,,,,,,,,
2026-05-01,1.07,0.58,4.72,5.0563,111.13846,7305306.0,1.07,4.72,4.1949
2026-06-01,1.12,0.16,4.64,5.1760,110.15775,7353293.0,1.12,4.64,4.2177
2026-07-01,1.22,0.07,4.44,5.0767,109.91611,7372243.0,1.22,4.44,4.0830
2026-08-01,1.09,-0.32,4.22,5.1810,NaN,NaN,1.09,4.22,4.2931
2026-09-01,0.62,NaN,NaN,NaN,NaN,NaN,0.57,NaN,NaN


---

# Fase 1 — Faixa de erro histórico do Focus (IPCA), por horizonte

Não é um modelo novo, é honestidade estatística: comparar a mediana do Focus, coletada
todo dia, com a inflação que de fato aconteceu nos meses seguintes. O objetivo é uma
frase do tipo *"a projeção mediana para o IPCA é X, e historicamente a previsão a doze
meses erra em mais ou menos Y pontos em 70% dos casos"* — sem tentar bater o Focus.

## 1.1 Coleta — expectativa de IPCA a 24 meses

A série de 12 meses (`df_focus_12m`) já foi coletada na Fase 0, com histórico desde 2001.
A de 24 meses só existe desde 2021 no serviço do BCB — entra como segundo horizonte de
comparação, com a ressalva de amostra mais curta.

In [12]:
def coletar_focus_inflacao_horizonte(entidade: str, indicador: str = "IPCA", suavizada: str = "N") -> pd.DataFrame:
    """
    Coleta uma série de expectativa de inflação de horizonte fixo do Focus (12 ou 24
    meses à frente, conforme `entidade`). Filtra baseCalculo=0 para uma linha por data.
    """
    filtro = f"Indicador eq '{indicador}' and Suavizada eq '{suavizada}' and baseCalculo eq 0"
    df = _consultar_focus_odata(entidade, filtro)
    df["Data"] = pd.to_datetime(df["Data"])
    return df.set_index("Data").sort_index()


try:
    df_focus_24m = coletar_focus_inflacao_horizonte("ExpectativasMercadoInflacao24Meses")
    salvar_parquet_seguro(df_focus_24m, RAW_DIR / "focus_ipca_24m.parquet")
    registrar_log("coleta_focus_24m", "ok", {"n_obs": len(df_focus_24m)})
except Exception as e:
    df_focus_24m = pd.DataFrame()
    registrar_log("coleta_focus_24m", "erro", {"mensagem": str(e)})

df_focus_24m[["Mediana"]].describe()

2026-09-18 03:54:14,567 | INFO | [coleta_focus_24m] ok | {'n_obs': 1370}


,Mediana
count,1370.000000
mean,3.850455
std,0.237047
min,3.335000
25%,3.669700
50%,3.802150
75%,3.991100
max,4.457200


## 1.2 Índice de preços acumulado (a partir do IPCA mensal)

Para saber quanto o IPCA realmente acumulou entre o mês da coleta e `h` meses depois,
construímos um índice de preços a partir da variação mensal do IPCA (já coletada na
Fase 0). A inflação realizada num intervalo é simplesmente a razão entre o índice no
fim e no início do intervalo.

In [13]:
def construir_indice_precos(ipca_mensal: pd.Series) -> pd.Series:
    """
    Constrói um índice de preços (base 100 no primeiro mês) a partir de uma série de
    variação mensal do IPCA em %. Meses sem dado ainda publicado (cauda da série) são
    descartados antes de acumular, para não interromper o índice no meio.
    """
    serie = ipca_mensal.dropna()
    descartados = ipca_mensal.index.difference(serie.index)
    if len(descartados) > 0:
        print(f"construir_indice_precos: descartando {len(descartados)} mês(es) sem IPCA publicado ainda: "
              f"{[d.date() for d in sorted(descartados)]}")
    fatores = 1 + serie / 100
    return 100 * fatores.cumprod()


indice_precos = construir_indice_precos(base["ipca_mensal"])
indice_precos.tail()

construir_indice_precos: descartando 1 mês(es) sem IPCA publicado ainda: [datetime.date(2026, 9, 1)]


data
2026-04-01    372.402439
2026-05-01    374.562373
2026-06-01    375.161673
2026-07-01    375.424286
2026-08-01    374.222928
Freq: MS, Name: ipca_mensal, dtype: float64

## 1.3 Erro do Focus por horizonte (previsto vs. realizado)

Atenção a uma armadilha específica desta série: `ExpectativasMercadoInflacao24Meses` **não**
é a inflação acumulada nos 24 meses seguintes — é a taxa esperada para o *segundo* ano à
frente (meses 12 a 24), sozinho. Confirmado empiricamente: comparar essa mediana contra a
inflação acumulada de 0 a 24 meses gera um erro médio de ~6,7 p.p. (viés absurdo para uma
previsão de mercado); comparando corretamente contra a inflação realizada só entre os meses
12 e 24, o erro médio cai para ~0,8 p.p. — nível compatível com o do horizonte de 12 meses.
Por isso a função abaixo recebe uma janela `(meses_inicio, meses_fim)` em vez de um único
horizonte: o de 12 meses usa (0, 12); o de 24 meses usa (12, 24).

In [14]:
def calcular_erro_focus(previsoes: pd.DataFrame, indice_precos: pd.Series, meses_inicio: int, meses_fim: int) -> pd.DataFrame:
    """
    Para cada data de coleta do Focus, compara a mediana prevista (inflação acumulada
    na janela [meses_inicio, meses_fim) a partir do mês de referência) com a inflação de
    fato realizada na mesma janela, medida pelo índice de preços. Descarta linhas cuja
    janela ainda não se completou (mês final além do último IPCA publicado) — não é
    erro, é previsão que ainda não pôde ser julgada.

    Retorna uma linha por coleta, com previsto, realizado, erro (realizado - previsto:
    positivo quando o Focus subestimou a inflação) e o rótulo do horizonte (meses_fim,
    usado só para identificar/agrupar).
    """
    df = previsoes[["Mediana"]].rename(columns={"Mediana": "previsto"}).copy()
    df["mes_referencia"] = df.index.to_period("M").to_timestamp()
    mes_inicio_janela = df["mes_referencia"] + pd.DateOffset(months=meses_inicio)
    mes_fim_janela = df["mes_referencia"] + pd.DateOffset(months=meses_fim)

    indice_inicio = indice_precos.reindex(mes_inicio_janela).to_numpy()
    indice_fim = indice_precos.reindex(mes_fim_janela).to_numpy()
    df["realizado"] = (indice_fim / indice_inicio - 1) * 100
    df["erro"] = df["realizado"] - df["previsto"]
    df["horizonte_meses"] = meses_fim

    df_completo = df.dropna(subset=["realizado"])
    n_descartadas = len(df) - len(df_completo)
    if n_descartadas > 0:
        print(f"calcular_erro_focus (horizonte {meses_fim}m): descartando {n_descartadas} coleta(s) cujo horizonte "
              f"ainda não se completou (mais recentes que {df_completo.index.max().date()}) — não é erro, é previsão futura.")

    return df_completo[["mes_referencia", "previsto", "realizado", "erro", "horizonte_meses"]]


df_erro_12m = calcular_erro_focus(df_focus_12m, indice_precos, meses_inicio=0, meses_fim=12)
df_erro_24m = calcular_erro_focus(df_focus_24m, indice_precos, meses_inicio=12, meses_fim=24)
df_erros_focus = pd.concat([df_erro_12m, df_erro_24m], ignore_index=True)

registrar_log("erro_focus_por_horizonte", "ok", {
    "n_obs_12m": len(df_erro_12m),
    "n_obs_24m": len(df_erro_24m),
})

df_erros_focus.groupby("horizonte_meses")["erro"].describe()

2026-09-18 03:54:14,609 | INFO | [erro_focus_por_horizonte] ok | {'n_obs_12m': 5685, 'n_obs_24m': 860}


calcular_erro_focus (horizonte 12m): descartando 551 coleta(s) cujo horizonte ainda não se completou (mais recentes que 2025-08-29) — não é erro, é previsão futura.
calcular_erro_focus (horizonte 24m): descartando 510 coleta(s) cujo horizonte ainda não se completou (mais recentes que 2024-08-30) — não é erro, é previsão futura.


,count,mean,std,min,25%,50%,75%,max
horizonte_meses,,,,,,,,
12,5685.0,0.624529,2.135644,-5.732783,-0.650863,0.379788,1.514199,8.244281
24,860.0,0.798109,0.450438,-0.467099,0.480053,0.897251,1.106637,1.515535


## 1.4 Faixa de incerteza — a frase final

`Y70` é o valor tal que, historicamente, o erro absoluto da mediana do Focus ficou
dentro de ± Y70 pontos em 70% das vezes (mesma leitura para 80%). Não é intervalo de
confiança formal — é a estatística mais simples que já responde à pergunta da mesa:
"o Focus está dizendo X, quanto isso pode errar na prática?"

In [15]:
def resumir_incerteza_focus(df_erros: pd.DataFrame, niveis: tuple[float, ...] = (0.70, 0.80, 0.90)) -> pd.DataFrame:
    """
    Resume, por horizonte, a faixa de erro absoluto do Focus nos níveis de confiança
    pedidos (ex.: erro absoluto ficou dentro de ± Y em 70% dos casos históricos).
    """
    linhas = []
    for horizonte, grupo in df_erros.groupby("horizonte_meses"):
        erro_abs = grupo["erro"].abs()
        linha = {
            "horizonte_meses": horizonte,
            "n_obs": len(grupo),
            "erro_medio": grupo["erro"].mean(),
            "erro_absoluto_medio": erro_abs.mean(),
        }
        for nivel in niveis:
            linha[f"faixa_{int(nivel*100)}pct"] = erro_abs.quantile(nivel)
        linhas.append(linha)
    return pd.DataFrame(linhas).set_index("horizonte_meses").round(2)


resumo_incerteza = resumir_incerteza_focus(df_erros_focus)
registrar_log("resumo_incerteza_focus", "ok", resumo_incerteza.to_dict(orient="index"))
resumo_incerteza

2026-09-18 03:54:14,635 | INFO | [resumo_incerteza_focus] ok | {12: {'n_obs': 5685, 'erro_medio': 0.62, 'erro_absoluto_medio': 1.56, 'faixa_70pct': 1.76, 'faixa_80pct': 2.38, 'faixa_90pct': 3.46}, 24: {'n_obs': 860, 'erro_medio': 0.8, 'erro_absoluto_medio': 0.82, 'faixa_70pct': 1.05, 'faixa_80pct': 1.16, 'faixa_90pct': 1.39}}


,n_obs,erro_medio,erro_absoluto_medio,faixa_70pct,faixa_80pct,faixa_90pct
horizonte_meses,,,,,,
12,5685,0.62,1.56,1.76,2.38,3.46
24,860,0.80,0.82,1.05,1.16,1.39


In [16]:
DESCRICAO_HORIZONTE = {
    12: "o IPCA acumulado nos próximos 12 meses",
    24: "o IPCA do segundo ano à frente (entre o 12º e o 24º mês)",
}


def frase_incerteza_focus(horizonte_meses: int, previsoes: pd.DataFrame, resumo: pd.DataFrame, nivel: float = 0.70) -> str:
    """
    Monta a frase de comunicação de incerteza para um horizonte: previsão atual do
    Focus + faixa histórica de erro absoluto no nível de confiança escolhido.
    """
    previsto_atual = previsoes["Mediana"].iloc[-1]
    data_previsao = previsoes.index[-1].strftime("%d/%m/%Y")
    faixa = resumo.loc[horizonte_meses, f"faixa_{int(nivel*100)}pct"]
    descricao = DESCRICAO_HORIZONTE[horizonte_meses]
    return (
        f"Em {data_previsao}, a projeção mediana do Focus para {descricao} é "
        f"{previsto_atual:.2f}%. Historicamente, essa previsão erra em mais ou menos "
        f"{faixa:.2f} pontos percentuais em {int(nivel*100)}% dos casos "
        f"(com base em {int(resumo.loc[horizonte_meses, 'n_obs'])} observações)."
    )


print(frase_incerteza_focus(12, df_focus_12m, resumo_incerteza))
print(frase_incerteza_focus(24, df_focus_24m, resumo_incerteza))

Em 11/09/2026, a projeção mediana do Focus para o IPCA acumulado nos próximos 12 meses é 4.65%. Historicamente, essa previsão erra em mais ou menos 1.76 pontos percentuais em 70% dos casos (com base em 5685 observações).
Em 11/09/2026, a projeção mediana do Focus para o IPCA do segundo ano à frente (entre o 12º e o 24º mês) é 3.93%. Historicamente, essa previsão erra em mais ou menos 1.05 pontos percentuais em 70% dos casos (com base em 860 observações).


---

# Fase 2 — Distribuição condicional de retorno por classe de ativo

Pergunta: dado o cenário atual de Selic e IPCA, qual a distribuição esperada de retorno
para os próximos meses? Usa o Focus como insumo (a Fase 1 já mede quanto confiar nele),
não compete com ele.

**Lacuna de dados conhecida:** as classes IPCA+ longo, prefixado e bolsa local dependem
de uma fonte de cotações que a casa ainda não definiu (o briefing aponta "possivelmente
Comdinheiro"). Testado neste ambiente: Yahoo Finance e Stooq estão inacessíveis (erro de
rede). Por isso a v1 usa apenas **CDI** (SGS) e **dólar** (câmbio, já coletado na Fase 0)
— dado real, não simulado. A metodologia é genérica: adicionar uma nova classe de ativo
mais tarde é só passar a série de retorno para as mesmas funções.

Técnicas, na ordem em que aparecem abaixo:
1. Regressão quantílica (quantis 10/50/90) condicionada ao cenário (nível e variação de
   Selic e IPCA).
2. Conformal prediction (CQR com janela de calibração móvel) para corrigir a cobertura
   do intervalo — validado com backtest em janela expansível, nunca embaralhado.
3. Bootstrap em blocos para simular a distribuição do retorno **acumulado em 12 meses**,
   preservando a dependência temporal dos resíduos.

In [17]:
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg

## 2.1 Cenário (X) e retorno-alvo (y), com alinhamento temporal correto

`X` no mês `t` usa apenas informação já publicada até o fim de `t` (nível de Selic e
IPCA 12m, e suas variações mensais). `y` no mês `t` é o retorno **realizado em `t+1`**
— por isso o `shift(-1)`. Isso garante que, ao treinar com dados até a linha `i`, o
modelo nunca viu informação do futuro em relação ao que está prevendo.

Todo modelo precisa que suas variáveis comecem e terminem juntas — cortar isso "na
unha" com um `dropna()` genérico funciona, mas esconde qual coluna mandou no corte.
Por isso o corte aqui é explícito: reporta a cobertura de cada variável antes de
juntar, identifica qual delas limita o início da amostra, e confirma que a janela
resultante é contígua (sem buraco escondido no meio, só corte nas bordas).

In [18]:
cenario = base_modelagem[["selic_over_mensal", "ipca_12m"]].copy()
cenario["delta_selic"] = cenario["selic_over_mensal"].diff()
cenario["delta_ipca_12m"] = cenario["ipca_12m"].diff()

retorno_dolar = base_modelagem["cambio_fim_mes"].pct_change() * 100
retornos = pd.DataFrame({
    "cdi": base_modelagem["cdi_mensal"],
    "dolar": retorno_dolar,
})

COLUNAS_X = ["selic_over_mensal", "ipca_12m", "delta_selic", "delta_ipca_12m"]
dados_completos = cenario.join(retornos.shift(-1), how="inner")

cobertura_modelo = pd.DataFrame({
    "primeira_data_valida": {c: dados_completos[c].dropna().index.min() for c in dados_completos.columns},
    "n_obs_validas": dados_completos.notna().sum(),
})
coluna_limitante = cobertura_modelo["primeira_data_valida"].idxmax()

dados_modelo = dados_completos.dropna()

# sanity check: a janela resultante deve ser contígua (corte só nas bordas) — um
# dropna() que remove linhas no MEIO da janela indicaria um buraco escondido numa
# das fontes, não o efeito esperado de alinhar datas de início diferentes
n_esperado = len(pd.date_range(dados_modelo.index.min(), dados_modelo.index.max(), freq="MS"))
if len(dados_modelo) != n_esperado:
    print(f"AVISO: {n_esperado - len(dados_modelo)} mês(es) faltando dentro da janela comum — não é só corte de borda, checar gaps internos por coluna.")

print(f"Cobertura por variável (antes do corte comum):")
print(cobertura_modelo)
print(f"\nJanela comum do modelo: {dados_modelo.index.min().date()} a {dados_modelo.index.max().date()} "
      f"({len(dados_modelo)} obs, contígua). Início definido por '{coluna_limitante}'.")
dados_modelo.tail()

Cobertura por variável (antes do corte comum):
                  primeira_data_valida  n_obs_validas
selic_over_mensal           2012-03-01            175
ipca_12m                    2012-03-01            174
delta_selic                 2012-04-01            174
delta_ipca_12m              2012-04-01            173
cdi                         2012-03-01            174
dolar                       2012-03-01            173

Janela comum do modelo: 2012-04-01 a 2026-07-01 (172 obs, contígua). Início definido por 'delta_selic'.


,selic_over_mensal,ipca_12m,delta_selic,delta_ipca_12m,cdi,dolar
data,,,,,,
2026-03-01,1.21,4.14,0.21,0.33,1.09,-4.422473
2026-04-01,1.09,4.39,-0.12,0.25,1.07,1.369286
2026-05-01,1.07,4.72,-0.02,0.33,1.12,2.367344
2026-06-01,1.12,4.64,0.05,-0.08,1.22,-1.918470
2026-07-01,1.22,4.44,0.10,-0.20,1.09,2.054484


## 2.2 Backtest walk-forward com regressão quantílica + conformal (CQR)

A cada mês do backtest: treina só com o passado (janela expansível), prevê os quantis
10/50/90 para o mês seguinte, e corrige o intervalo [q10, q90] com os erros de
conformidade dos últimos `janela_calibracao` meses (nunca usando o próprio mês testado
— por isso a correção só existe depois que a janela de calibração está cheia).

`janela_minima=60` e `janela_calibracao=24` (5 e 2 anos) — menores que um valor mais
"padrão" de 120/36 de propósito: com a amostra cortada em 2012-03 (~171 meses), usar
120+36 deixaria só ~15 observações para checar calibração, estatisticamente pouco
confiável. Com 60+24, restam ~87 — ainda pouco para um MFO, mas o suficiente para o
número de cobertura empírica não ser ruído puro.

In [19]:
def backtest_walk_forward_cqr(
    dados: pd.DataFrame,
    colunas_x: list[str],
    coluna_y: str,
    q_low: float = 0.1,
    q_high: float = 0.9,
    janela_minima: int = 60,
    janela_calibracao: int = 24,
) -> pd.DataFrame:
    """
    Backtest de regressão quantílica (quantis q_low/mediana/q_high) em janela
    expansível, com correção conformal (CQR) por janela de calibração móvel.

    Retorna uma linha por mês testado, com a previsão bruta, a previsão corrigida
    pelo conformal e o valor realizado — a base para checar calibração depois.
    """
    dados = dados.sort_index()
    X = sm.add_constant(dados[colunas_x])
    y = dados[coluna_y]
    nivel_cobertura_nominal = q_high - q_low

    registros = []
    scores_conformidade = []

    for i in range(janela_minima, len(dados)):
        X_treino, y_treino = X.iloc[:i], y.iloc[:i]
        X_teste = X.iloc[[i]]
        y_real = y.iloc[i]

        preds = {}
        for q in (q_low, 0.5, q_high):
            resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)
            preds[q] = resultado.predict(X_teste).iloc[0]

        if len(scores_conformidade) >= janela_calibracao:
            correcao = max(np.quantile(scores_conformidade[-janela_calibracao:], nivel_cobertura_nominal), 0.0)
            q_low_cqr, q_high_cqr = preds[q_low] - correcao, preds[q_high] + correcao
        else:
            q_low_cqr, q_high_cqr = np.nan, np.nan

        registros.append({
            "data": dados.index[i],
            "y_real": y_real,
            "q_low_bruto": preds[q_low],
            "q_mediana": preds[0.5],
            "q_high_bruto": preds[q_high],
            "q_low_cqr": q_low_cqr,
            "q_high_cqr": q_high_cqr,
        })

        # score de não-conformidade (CQR): o quanto o valor real, se algum dia soubéssemos,
        # ficaria fora do intervalo bruto — usado só para calibrar passos FUTUROS
        scores_conformidade.append(max(preds[q_low] - y_real, y_real - preds[q_high]))

    return pd.DataFrame(registros).set_index("data")


resultado_cdi = backtest_walk_forward_cqr(dados_modelo, COLUNAS_X, "cdi")
resultado_dolar = backtest_walk_forward_cqr(dados_modelo, COLUNAS_X, "dolar")

registrar_log("backtest_cqr", "ok", {"n_cdi": len(resultado_cdi), "n_dolar": len(resultado_dolar)})
resultado_dolar.tail()

/tmp/ipykernel_1645/3598986831.py:32: IterationLimitWarning: Maximum number of iterations (2000) reached.
  resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)


/tmp/ipykernel_1645/3598986831.py:32: IterationLimitWarning: Maximum number of iterations (2000) reached.
  resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)


/tmp/ipykernel_1645/3598986831.py:32: IterationLimitWarning: Maximum number of iterations (2000) reached.
  resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)


/tmp/ipykernel_1645/3598986831.py:32: IterationLimitWarning: Maximum number of iterations (2000) reached.
  resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)


/tmp/ipykernel_1645/3598986831.py:32: IterationLimitWarning: Maximum number of iterations (2000) reached.
  resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)


/tmp/ipykernel_1645/3598986831.py:32: IterationLimitWarning: Maximum number of iterations (2000) reached.
  resultado = QuantReg(y_treino, X_treino).fit(q=q, max_iter=2000)


2026-09-18 03:54:27,465 | INFO | [backtest_cqr] ok | {'n_cdi': 112, 'n_dolar': 112}


,y_real,q_low_bruto,q_mediana,q_high_bruto,q_low_cqr,q_high_cqr
data,,,,,,
2026-03-01,-4.422473,-5.453969,-0.116392,2.178491,-5.453969,2.178491
2026-04-01,1.369286,-2.660388,1.263368,5.423452,-2.660388,5.423452
2026-05-01,2.367344,-3.581542,0.810857,5.070233,-3.581542,5.070233
2026-06-01,-1.918470,-4.032444,0.091732,3.680169,-4.032444,3.680169
2026-07-01,2.054484,-4.287885,-0.370264,2.554945,-4.287885,2.554945


## 2.3 Verificação de calibração (validação obrigatória)

O intervalo bruto da regressão quantílica (10/90) é nominalmente de 80%. Ele quase
nunca acerta esse número na prática — é exatamente por isso que existe o conformal.
Comparamos os dois: cobertura empírica do intervalo bruto vs. do intervalo corrigido
pelo CQR. O corrigido precisa ficar perto de 80%; se não ficar, a janela de calibração
ou o modelo base precisam ser revistos antes de qualquer uso real.

In [20]:
def checar_calibracao(resultado_backtest: pd.DataFrame) -> dict:
    """
    Mede a cobertura empírica dos intervalos bruto e corrigido (CQR) sobre o período
    de backtest em que a correção já estava disponível (janela de calibração cheia).
    """
    df = resultado_backtest.dropna(subset=["q_low_cqr", "q_high_cqr"])
    n_descartadas = len(resultado_backtest) - len(df)
    if n_descartadas > 0:
        print(f"checar_calibracao: descartando {n_descartadas} mês(es) iniciais do backtest "
              f"(janela de calibração ainda não estava cheia) — restam {len(df)} obs válidas.")
    cobertura_bruta = ((df["y_real"] >= df["q_low_bruto"]) & (df["y_real"] <= df["q_high_bruto"])).mean()
    cobertura_cqr = ((df["y_real"] >= df["q_low_cqr"]) & (df["y_real"] <= df["q_high_cqr"])).mean()
    largura_media_bruta = (df["q_high_bruto"] - df["q_low_bruto"]).mean()
    largura_media_cqr = (df["q_high_cqr"] - df["q_low_cqr"]).mean()
    return {
        "n_obs_validas": len(df),
        "cobertura_nominal": 0.80,
        "cobertura_empirica_bruta": round(cobertura_bruta, 3),
        "cobertura_empirica_cqr": round(cobertura_cqr, 3),
        "largura_media_bruta_pp": round(largura_media_bruta, 2),
        "largura_media_cqr_pp": round(largura_media_cqr, 2),
    }


calibracao_cdi = checar_calibracao(resultado_cdi)
calibracao_dolar = checar_calibracao(resultado_dolar)
registrar_log("calibracao_cqr", "ok", {"cdi": calibracao_cdi, "dolar": calibracao_dolar})

pd.DataFrame({"cdi": calibracao_cdi, "dolar": calibracao_dolar}).T

2026-09-18 03:54:27,487 | INFO | [calibracao_cqr] ok | {'cdi': {'n_obs_validas': 88, 'cobertura_nominal': 0.8, 'cobertura_empirica_bruta': np.float64(0.693), 'cobertura_empirica_cqr': np.float64(0.739), 'largura_media_bruta_pp': np.float64(0.18), 'largura_media_cqr_pp': np.float64(0.21)}, 'dolar': {'n_obs_validas': 88, 'cobertura_nominal': 0.8, 'cobertura_empirica_bruta': np.float64(0.852), 'cobertura_empirica_cqr': np.float64(0.886), 'largura_media_bruta_pp': np.float64(10.98), 'largura_media_cqr_pp': np.float64(12.51)}}


checar_calibracao: descartando 24 mês(es) iniciais do backtest (janela de calibração ainda não estava cheia) — restam 88 obs válidas.
checar_calibracao: descartando 24 mês(es) iniciais do backtest (janela de calibração ainda não estava cheia) — restam 88 obs válidas.


,n_obs_validas,cobertura_nominal,cobertura_empirica_bruta,cobertura_empirica_cqr,largura_media_bruta_pp,largura_media_cqr_pp
cdi,88.0,0.8,0.693,0.739,0.18,0.21
dolar,88.0,0.8,0.852,0.886,10.98,12.51


## 2.4 Cenário atual → distribuição de retorno acumulado em 12 meses (bootstrap em blocos)

A regressão quantílica walk-forward responde "mês que vem". Para uso na mesa importa
mais o acumulado de 12 meses. Em vez de assumir independência mês a mês (o que
subestimaria a incerteza), encadeamos blocos contíguos de resíduos históricos — isso
preserva a autocorrelação de curto prazo da série — somados à mediana condicional do
cenário atual, e compomos o retorno resultante.

In [21]:
def bootstrap_blocos_retorno_acumulado(
    residuos: np.ndarray,
    mediana_condicional: float,
    horizonte_meses: int = 12,
    tamanho_bloco: int = 3,
    n_simulacoes: int = 5000,
    seed: int = 42,
) -> np.ndarray:
    """
    Simula `n_simulacoes` trajetórias do retorno acumulado em `horizonte_meses`,
    encadeando blocos contíguos de `residuos` históricos (bootstrap em blocos, preserva
    dependência temporal de curto prazo) e somando à mediana condicional prevista para
    o cenário atual — assume que o cenário (e por isso a mediana) se mantém ao longo do
    horizonte, uma simplificação razoável para uma v1.
    """
    rng = np.random.default_rng(seed)
    n = len(residuos)
    simulacoes = np.empty(n_simulacoes)

    for s in range(n_simulacoes):
        retornos_simulados = []
        while len(retornos_simulados) < horizonte_meses:
            inicio = rng.integers(0, n - tamanho_bloco + 1)
            bloco = residuos[inicio: inicio + tamanho_bloco]
            retornos_simulados.extend((mediana_condicional + bloco).tolist())
        fator_acumulado = np.prod(1 + np.array(retornos_simulados[:horizonte_meses]) / 100)
        simulacoes[s] = (fator_acumulado - 1) * 100

    return simulacoes


def distribuicao_12m_cenario_atual(dados: pd.DataFrame, colunas_x: list[str], coluna_y: str) -> dict:
    """
    Ajusta a regressão quantílica com todo o histórico disponível, extrai os resíduos
    do modelo de mediana, e usa bootstrap em blocos para simular a distribuição do
    retorno acumulado em 12 meses a partir do cenário (Selic/IPCA) mais recente.
    """
    X = sm.add_constant(dados[colunas_x])
    y = dados[coluna_y]

    modelo_mediana = QuantReg(y, X).fit(q=0.5, max_iter=2000)
    residuos = (y - modelo_mediana.predict(X)).to_numpy()

    cenario_atual = X.iloc[[-1]]
    mediana_atual = modelo_mediana.predict(cenario_atual).iloc[0]

    simulacoes = bootstrap_blocos_retorno_acumulado(residuos, mediana_atual)

    return {
        "data_cenario": dados.index[-1],
        "mediana_mensal_condicional": mediana_atual,
        "q10_12m": np.quantile(simulacoes, 0.10),
        "q50_12m": np.quantile(simulacoes, 0.50),
        "q90_12m": np.quantile(simulacoes, 0.90),
    }


distribuicao_cdi_12m = distribuicao_12m_cenario_atual(dados_modelo, COLUNAS_X, "cdi")
distribuicao_dolar_12m = distribuicao_12m_cenario_atual(dados_modelo, COLUNAS_X, "dolar")

registrar_log("distribuicao_12m", "ok", {
    "cdi": {k: (str(v) if k == "data_cenario" else round(v, 2)) for k, v in distribuicao_cdi_12m.items()},
    "dolar": {k: (str(v) if k == "data_cenario" else round(v, 2)) for k, v in distribuicao_dolar_12m.items()},
})

pd.DataFrame({"cdi": distribuicao_cdi_12m, "dolar": distribuicao_dolar_12m}).T

2026-09-18 03:54:27,735 | INFO | [distribuicao_12m] ok | {'cdi': {'data_cenario': '2026-07-01 00:00:00', 'mediana_mensal_condicional': np.float64(1.13), 'q10_12m': np.float64(14.25), 'q50_12m': np.float64(14.55), 'q90_12m': np.float64(14.86)}, 'dolar': {'data_cenario': '2026-07-01 00:00:00', 'mediana_mensal_condicional': np.float64(-0.12), 'q10_12m': np.float64(-16.16), 'q50_12m': np.float64(-0.33), 'q90_12m': np.float64(20.97)}}


,data_cenario,mediana_mensal_condicional,q10_12m,q50_12m,q90_12m
cdi,2026-07-01 00:00:00,1.132464,14.24549,14.547614,14.863944
dolar,2026-07-01 00:00:00,-0.123856,-16.157984,-0.330912,20.973714


## 2.5 Relatório consolidado por classe de ativo

In [22]:
def relatorio_classe_ativo(nome_classe: str, resultado_backtest: pd.DataFrame, calibracao: dict, distribuicao_12m: dict) -> str:
    """Monta o texto de saída da Fase 2 para uma classe de ativo, no estilo da frase da Fase 1."""
    ultima_previsao = resultado_backtest.dropna(subset=["q_low_cqr", "q_high_cqr"]).iloc[-1]
    return (
        f"[{nome_classe.upper()}] Cenário de {distribuicao_12m['data_cenario'].strftime('%m/%Y')}: "
        f"retorno esperado no próximo mês entre {ultima_previsao['q_low_cqr']:.2f}% e "
        f"{ultima_previsao['q_high_cqr']:.2f}% (80% de confiança, calibração histórica "
        f"empírica de {calibracao['cobertura_empirica_cqr']*100:.0f}%). "
        f"Acumulado em 12 meses: entre {distribuicao_12m['q10_12m']:.2f}% e "
        f"{distribuicao_12m['q90_12m']:.2f}%, mediana {distribuicao_12m['q50_12m']:.2f}% "
        f"(bootstrap em blocos, cenário atual mantido constante)."
    )


print(relatorio_classe_ativo("CDI", resultado_cdi, calibracao_cdi, distribuicao_cdi_12m))
print(relatorio_classe_ativo("Dólar", resultado_dolar, calibracao_dolar, distribuicao_dolar_12m))

[CDI] Cenário de 07/2026: retorno esperado no próximo mês entre 0.95% e 1.28% (80% de confiança, calibração histórica empírica de 74%). Acumulado em 12 meses: entre 14.25% e 14.86%, mediana 14.55% (bootstrap em blocos, cenário atual mantido constante).
[DÓLAR] Cenário de 07/2026: retorno esperado no próximo mês entre -4.29% e 2.55% (80% de confiança, calibração histórica empírica de 89%). Acumulado em 12 meses: entre -16.16% e 20.97%, mediana -0.33% (bootstrap em blocos, cenário atual mantido constante).


---

# Fase 3 — Black-Litterman adaptado a family office

Ponte formal entre a alocação estratégica da casa e a visão tática do comitê. A
adaptação central em relação ao Black-Litterman de livro-texto: **não existe peso de
mercado** para classe de ativo numa carteira de family office. O prior deixa de ser
"o mercado" (`w_mkt`) e passa a ser a política de investimento da GHIA (`w_ref`) — o
retorno de equilíbrio implícito é calculado em cima dessa política, não de um índice.

**Escopo desta v1** (conforme o briefing): poucas classes de ativo, uma única visão,
output mostrando pesos antes e depois. Sem dashboard, sem restrição sofisticada, sem
cliente individual. O argumento de venda não é a carteira sugerida — é o registro da
visão para comparar, alguns meses depois, se ela agregou ou destruiu valor (Seção 3.7).

**Lacuna de dados**: IPCA+ longo, prefixado e ilíquidos (imobiliário, PE) continuam sem
fonte (mesma pendência da Fase 2 — Comdinheiro ou equivalente). Esta v1 roda com os
três ativos para os quais existe dado real: CDI, dólar e bolsa local (Ibovespa, nova
coleta abaixo). A função de desmoothing de Geltner é implementada e validada com uma
série sintética (Seção 3.3), pronta para quando um retorno de ilíquido real chegar —
sem isso, o otimizador trataria a suavização artificial desses ativos como se fosse
baixo risco de verdade, e concentraria a carteira neles por um motivo espúrio.

## 3.0 Coleta adicional — Ibovespa (bolsa local)

O SGS chegou a publicar o Ibovespa (código 7845) — confirmado batendo exatamente com
os fechamentos reais de dez/2008 (37.550 pontos) e dez/2015 (43.349 pontos) — mas
descontinuou a série em 08/2019. Sem alternativa gratuita no BCB/IBGE para o índice
corrente, a coleta usa Yahoo Finance (ticker `^BVSP`), como sugerido. Rede bloqueada
neste ambiente de teste (mesmo problema já visto com Stooq na Fase 2) — deve funcionar
localmente; se falhar, a Fase 3 roda só com CDI e dólar.

In [23]:
import yfinance as yf
from sklearn.covariance import LedoitWolf
from pypfopt import BlackLittermanModel


def coletar_ibovespa(data_inicio: str = DATA_INICIO_MODELAGEM) -> pd.Series:
    """
    Coleta o fechamento mensal do Ibovespa via Yahoo Finance e retorna o retorno
    percentual mensal. yfinance às vezes devolve um DataFrame vazio em vez de lançar
    exceção numa falha de rede — por isso o erro é levantado explicitamente aqui, para
    que `executar_com_retentativas` reconheça a falha e tente de novo.
    """
    precos = yf.download("^BVSP", start=data_inicio, progress=False, auto_adjust=False)["Close"]
    if precos.empty:
        raise RuntimeError("yfinance devolveu série vazia para ^BVSP")
    precos_mensais = precos.resample("MS").last()
    retorno = precos_mensais.pct_change() * 100
    retorno.name = "bolsa"
    return retorno


try:
    retorno_bolsa = executar_com_retentativas(coletar_ibovespa)
    salvar_parquet_seguro(retorno_bolsa.to_frame(), RAW_DIR / "ibovespa.parquet")
    registrar_log("coleta_ibovespa", "ok", {"n_obs": len(retorno_bolsa)})
except Exception as e:
    retorno_bolsa = pd.Series(dtype=float, name="retorno_bolsa")
    registrar_log("coleta_ibovespa", "erro", {"mensagem": str(e)})
    print(f"Aviso: coleta do Ibovespa falhou ({e}). Fase 3 segue só com CDI e dólar.")

retorno_bolsa.tail()

2026-09-18 03:54:28,881 | WARNING | [coleta_ibovespa] erro | {'mensagem': "'DataFrame' object has no attribute 'to_frame'"}


Aviso: coleta do Ibovespa falhou ('DataFrame' object has no attribute 'to_frame'). Fase 3 segue só com CDI e dólar.


Series([], Name: retorno_bolsa, dtype: float64)

## 3.0.1 Proxy de prefixado e IPCA+ longo via curva de juros

Nem prefixado (LTN/NTN-F) nem IPCA+ longo (NTN-B) têm fonte de cotação direta ainda.
Os dois podem ser aproximados a partir da mesma série: o swap DI x Pré de 360 dias
(SGS 1178) — confirmado batendo com a Selic de 2% a.a. do corte histórico de 2020,
única série de curva de juros encontrada com fonte gratuita e confiável neste teste
(não achei um endpoint público para a curva completa de DI futuro nem para a ETTJ da
ANBIMA — só esse ponto de 360 dias).

**Prefixado**: reage à curva NOMINAL — usa o swap direto.
**IPCA+ longo**: reage à curva REAL — aproximada por Fisher, `juro_real ≈ juro_nominal
- expectativa_IPCA_Focus` (já coletada na Fase 1).

Em ambos os casos, variação de juro vira retorno de preço pela aproximação padrão de
duration: `ΔP/P ≈ -duration × Δy`. **As durations abaixo (2 anos para prefixado, 8
para IPCA+ longo) são valores ilustrativos de mercado, não a duration real de nenhum
título específico da carteira — ajustar depois.** Limitações desta proxy, para deixar
claro no que ela não é confiável: um único ponto de 360 dias não captura a forma toda
da curva (nível, inclinação, curvatura), Fisher é uma aproximação, e duration ignora
convexidade e efeito de rolagem. Serve para destravar a v1, não para precificar nada.

In [24]:
DURATION_PREFIXADO_ANOS = 2.0
DURATION_IPCA_LONGO_ANOS = 8.0


def coletar_swap_di_pre_360() -> pd.Series:
    """
    Coleta o swap DI x Pré de 360 dias (SGS 1178) e devolve a série mensal (fim de mês).
    `_coletar_serie_sgs` já tenta de novo por conta própria a cada janela de 9 anos —
    por isso esta função não é chamada de novo dentro de outra camada de retentativas
    (evita retentar em cascata: 4 tentativas aqui x 4 lá dentro vira até 16 chamadas).
    """
    serie_diaria = _coletar_serie_sgs("swap_di_pre_360", 1178, DATA_INICIO_MODELAGEM, diaria=True)
    return serie_diaria["swap_di_pre_360"].resample("MS").last()


try:
    swap_di_pre_360 = coletar_swap_di_pre_360()
    registrar_log("coleta_swap_di_pre", "ok", {"n_obs": len(swap_di_pre_360)})
except Exception as e:
    swap_di_pre_360 = pd.Series(dtype=float)
    registrar_log("coleta_swap_di_pre", "erro", {"mensagem": str(e)})
    print(f"Aviso: coleta do swap DI x Pré falhou ({e}). Sem proxy de prefixado/IPCA+ longo nesta rodada.")

focus_ipca_12m_mensal = df_focus_12m["Mediana"].resample("MS").last()

juro_real_proxy = swap_di_pre_360 - focus_ipca_12m_mensal  # Fisher aproximado

retorno_prefixado_proxy = (-DURATION_PREFIXADO_ANOS * swap_di_pre_360.diff()).rename("prefixado")
retorno_ipca_longo_proxy = (-DURATION_IPCA_LONGO_ANOS * juro_real_proxy.diff()).rename("ipca_longo")

pd.DataFrame({"swap_nominal": swap_di_pre_360, "juro_real_proxy": juro_real_proxy,
              "retorno_prefixado_proxy": retorno_prefixado_proxy, "retorno_ipca_longo_proxy": retorno_ipca_longo_proxy}).tail()

2026-09-18 03:54:29,852 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.1178/dados?formato=json&dataInicial=01%2F03%2F2012&dataFinal=28%2F02%2F2021 "HTTP/1.1 200 OK"


2026-09-18 03:54:30,813 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.1178/dados?formato=json&dataInicial=01%2F03%2F2021&dataFinal=18%2F09%2F2026 "HTTP/1.1 200 OK"


2026-09-18 03:54:31,003 | INFO | [coleta_swap_di_pre] ok | {'n_obs': 175}


,swap_nominal,juro_real_proxy,retorno_prefixado_proxy,retorno_ipca_longo_proxy
Date,,,,
2026-05-01,14.40,10.2252,-0.0,-0.7640
2026-06-01,14.15,9.9600,0.5,2.1216
2026-07-01,14.15,10.0384,-0.0,-0.6272
2026-08-01,13.90,9.5890,0.5,3.5952
2026-09-01,13.65,8.9952,0.5,4.7504


## 3.1 Retornos mensais dos ativos disponíveis

Mesmo cuidado de alinhamento da Seção 6.2: reporta a cobertura de cada ativo antes de
juntar e confirma que a janela final é contígua.

In [25]:
retornos_ativos = pd.DataFrame({
    "cdi": base_modelagem["cdi_mensal"],
    "dolar": base_modelagem["cambio_fim_mes"].pct_change() * 100,
})
if not retorno_bolsa.empty:
    retornos_ativos = retornos_ativos.join(retorno_bolsa, how="left")
if not swap_di_pre_360.empty:
    retornos_ativos = retornos_ativos.join(retorno_prefixado_proxy, how="left").join(retorno_ipca_longo_proxy, how="left")

cobertura_ativos = pd.DataFrame({
    "primeira_data_valida": {c: retornos_ativos[c].dropna().index.min() for c in retornos_ativos.columns},
    "n_obs_validas": retornos_ativos.notna().sum(),
})
print("Cobertura por ativo (antes do corte comum):")
print(cobertura_ativos)

retornos_ativos = retornos_ativos.dropna()
n_esperado = len(pd.date_range(retornos_ativos.index.min(), retornos_ativos.index.max(), freq="MS"))
if len(retornos_ativos) != n_esperado:
    print(f"AVISO: {n_esperado - len(retornos_ativos)} mês(es) faltando dentro da janela comum — checar gaps internos.")

print(f"\n{len(retornos_ativos)} observações mensais, de {retornos_ativos.index.min().date()} a {retornos_ativos.index.max().date()}.")
ATIVOS = list(retornos_ativos.columns)
retornos_ativos.tail()

Cobertura por ativo (antes do corte comum):
           primeira_data_valida  n_obs_validas
cdi                  2012-03-01            175
dolar                2012-04-01            173
prefixado            2012-04-01            174
ipca_longo           2012-04-01            174

173 observações mensais, de 2012-04-01 a 2026-08-01.


,cdi,dolar,prefixado,ipca_longo
data,,,,
2026-04-01,1.09,-4.422473,0.5,2.2784
2026-05-01,1.07,1.369286,-0.0,-0.7640
2026-06-01,1.12,2.367344,0.5,2.1216
2026-07-01,1.22,-1.918470,-0.0,-0.6272
2026-08-01,1.09,2.054484,0.5,3.5952


## 3.2 Covariância com encolhimento de Ledoit-Wolf

Covariância amostral pura é instável com poucas observações (armadilha citada no
briefing). O encolhimento de Ledoit-Wolf empurra a matriz amostral em direção a uma
estrutura mais simples, proporcionalmente a um coeficiente estimado dos próprios
dados. Anualizada (×12) porque `delta` e as visões nas Seções 3.4-3.5 são expressas em
base anual, convenção usual do Black-Litterman.

**Detalhe que importa aqui**: `sklearn.covariance.LedoitWolf` aplicado direto na
covariância bruta encolhe cada variância em direção à *média das variâncias de todos
os ativos da matriz*. Testado: com CDI (variância mensal ~0,09) e dólar (~18, 200x
maior) na mesma matriz, isso inflava a variância do CDI em ~4x mesmo com um
coeficiente de encolhimento pequeno (3%) — a diferença de escala entre os ativos é
grande demais para esse alvo de encolhimento fazer sentido. Correção: padronizar os
retornos (z-score) antes de encolher, para que o Ledoit-Wolf atue só sobre a
**correlação**, e recombinar com os desvios-padrão originais (não encolhidos) depois —
preserva a escala própria de cada ativo, só ajusta o quanto confiar na correlação
estimada entre eles.

In [26]:
def estimar_covariancia_ledoit_wolf(retornos: pd.DataFrame) -> tuple[pd.DataFrame, float]:
    """
    Estima a matriz de covariância anualizada encolhendo a CORRELAÇÃO (não a covariância
    bruta) via Ledoit-Wolf, e recombina com os desvios-padrão amostrais originais. Ver a
    explicação completa no markdown acima sobre por que encolher a covariância bruta
    direto distorce ativos de escala muito diferente (ex.: CDI vs. câmbio).

    Retorna a matriz de covariância e o coeficiente de encolhimento estimado (0 =
    correlação amostral pura, 1 = totalmente encolhida para a correlação nula/identidade).
    """
    desvios = retornos.std()
    lw = LedoitWolf().fit(retornos / desvios)
    corr_encolhida = lw.covariance_
    d = np.diag(desvios)
    cov_mensal = pd.DataFrame(d @ corr_encolhida @ d, index=retornos.columns, columns=retornos.columns)
    return cov_mensal * 12, lw.shrinkage_


cov_anual, coeficiente_encolhimento = estimar_covariancia_ledoit_wolf(retornos_ativos)
print(f"Coeficiente de encolhimento (na correlação): {coeficiente_encolhimento:.2f}")
cov_anual

Coeficiente de encolhimento (na correlação): 0.13


,cdi,dolar,prefixado,ipca_longo
cdi,1.078681,-1.266711,-0.142641,-1.628680
dolar,-1.266711,212.480141,1.687385,3.293294
prefixado,-0.142641,1.687385,8.785135,28.475624
ipca_longo,-1.628680,3.293294,28.475624,199.581193


## 3.3 Desmoothing de Geltner (utilitário, sem dado ilíquido ainda)

Retorno suavizado subestima volatilidade e correlação — sem corrigir isso, o
otimizador concentraria em ilíquido achando que é almoço grátis. Sem dado real de
ilíquido nesta v1, a função é validada com uma série sintética: gera um retorno
"verdadeiro" aleatório, suaviza artificialmente (simulando o efeito de marcação por
avaliação pouco frequente), desmoothing, e confere se recupera algo parecido com o
verdadeiro — prova de que a função funciona antes de precisar dela de verdade.

In [27]:
def desmoothing_geltner(retorno_suavizado: pd.Series, phi: float) -> pd.Series:
    """
    Reverte a suavização de um retorno observado (comum em ativos avaliados por marcação
    pouco frequente — imóveis, PE, crédito estruturado) assumindo que o valor suavizado
    observado é uma média móvel AR(1) do retorno verdadeiro:
    r_suavizado[t] = (1 - phi) * r_verdadeiro[t] + phi * r_suavizado[t-1].
    Invertendo: r_verdadeiro[t] = (r_suavizado[t] - phi * r_suavizado[t-1]) / (1 - phi).
    `phi` (0 a 1) mede o quanto o valor é suavizado; phi=0 significa "não suavizado" e
    devolve a série original.
    """
    return (retorno_suavizado - phi * retorno_suavizado.shift(1)) / (1 - phi)


def validar_desmoothing_geltner(n_meses: int = 240, phi: float = 0.6, seed: int = 42) -> float:
    """Testa desmoothing_geltner numa série sintética e retorna a correlação recuperada."""
    rng = np.random.default_rng(seed)
    retorno_verdadeiro = pd.Series(rng.normal(0.5, 3.0, n_meses))
    retorno_suavizado = retorno_verdadeiro.copy()
    for t in range(1, n_meses):
        retorno_suavizado.iloc[t] = (1 - phi) * retorno_verdadeiro.iloc[t] + phi * retorno_suavizado.iloc[t - 1]

    retorno_recuperado = desmoothing_geltner(retorno_suavizado, phi).dropna()
    correlacao = retorno_recuperado.corr(retorno_verdadeiro.iloc[1:])
    print(f"Volatilidade: verdadeira={retorno_verdadeiro.std():.2f}, suavizada={retorno_suavizado.std():.2f} "
          f"(subestimada, como esperado), recuperada={retorno_recuperado.std():.2f}")
    print(f"Correlação entre recuperado e verdadeiro: {correlacao:.3f} (esperado: alta, próxima de 1)")
    return correlacao


validar_desmoothing_geltner()

Volatilidade: verdadeira=2.70, suavizada=1.38 (subestimada, como esperado), recuperada=2.71
Correlação entre recuperado e verdadeiro: 1.000 (esperado: alta, próxima de 1)


np.float64(0.9999999999999997)

## 3.4 Retorno de equilíbrio implícito (prior = política da GHIA)

`Π = δ Σ w_ref` — igual ao Black-Litterman de livro-texto, mas `w_ref` é a alocação
estratégica da casa, não peso de mercado (que não existe aqui). **`W_REF_ILUSTRATIVO`
abaixo é um placeholder de exemplo — substituir pela política real da GHIA por perfil
de cliente antes de qualquer uso alem de prototipagem.**

`delta` (aversão a risco) **não pode** usar o valor-padrão de livro-texto (2,5,
calibrado para uma carteira de mercado diversificada — ações globais, baixa vol
relativa). Testado: com só 2-3 classes e o dólar carregando uma variância isolada
enorme, `delta=2.5` gera um Π de dólar de ~70% a.a. — sem sentido.

Primeira tentativa: calibrar pela fórmula clássica `δ = (E[R_ref] - Rf) / Var(R_ref)`
usando a média histórica de excesso de retorno da carteira de referência. Também
quebrou — testado: nesta amostra (~14 anos), o dólar se comporta perto de um passeio
aleatório contra o CDI, e a média histórica de excesso saiu negativa, o que devolve um
delta negativo (inválido). Usar uma média histórica ruidosa para calibrar delta é
irônico, ainda por cima: o Black-Litterman existe exatamente para não depender de
médias históricas de retorno, que são estimadores muito instáveis numa amostra curta.

Solução: `delta = prêmio_de_risco_ASSUMIDO / Var(R_ref)`. A variância histórica é um
estimador razoavelmente estável (ao contrário da média) e continua vindo dos dados
reais; o prêmio de risco vira um parâmetro explícito e assumido — não estimado — que
a mesa pode discutir e ajustar (3% a.a. abaixo é um valor moderado, conservador, para
uma carteira dominada por CDI; sem relação com a política real da GHIA ainda).

In [28]:
PESOS_ILUSTRATIVOS_POR_ORDEM = [0.50, 0.10, 0.15, 0.15, 0.10]  # cdi, dolar, bolsa, prefixado, ipca_longo (nessa ordem, se presentes)
W_REF_ILUSTRATIVO = pd.Series({a: p for a, p in zip(ATIVOS, PESOS_ILUSTRATIVOS_POR_ORDEM)}, name="peso_estrategico")
W_REF_ILUSTRATIVO = W_REF_ILUSTRATIVO / W_REF_ILUSTRATIVO.sum()  # garante soma 1 mesmo se algum ativo faltar (ex.: sem bolsa)

PREMIO_RISCO_ASSUMIDO_PP = 3.0  # a.a., parâmetro explícito — discutir com a mesa, não estimado dos dados


def calibrar_delta_aversao_risco(retornos: pd.DataFrame, pesos: pd.Series, premio_risco_assumido_pp: float) -> float:
    """
    Calibra a aversão a risco como delta = premio_risco_assumido / Var(R_ref), usando a
    variância HISTÓRICA da carteira de referência (estimador estável) mas um prêmio de
    risco ASSUMIDO em vez de estimado pela média histórica — que se mostrou instável
    demais nesta amostra (chegou a ficar negativa). Ver a explicação completa no
    markdown acima.
    """
    retorno_ref_mensal = (retornos * pesos).sum(axis=1)
    variancia_anual = retorno_ref_mensal.var() * 12
    return premio_risco_assumido_pp / variancia_anual


DELTA_AVERSAO_RISCO = calibrar_delta_aversao_risco(retornos_ativos, W_REF_ILUSTRATIVO, PREMIO_RISCO_ASSUMIDO_PP)
print(f"Delta calibrado (prêmio assumido {PREMIO_RISCO_ASSUMIDO_PP:.1f}% a.a. / variância histórica da carteira de referência): {DELTA_AVERSAO_RISCO:.2f}")

pi_equilibrio = DELTA_AVERSAO_RISCO * cov_anual.values @ W_REF_ILUSTRATIVO.values
pi_equilibrio = pd.Series(pi_equilibrio, index=ATIVOS, name="retorno_equilibrio_anual_pct")

print("\nPesos estratégicos (w_ref, ILUSTRATIVO — substituir pela política real da GHIA):")
print(W_REF_ILUSTRATIVO)
print("\nRetorno de equilíbrio implícito (Π), anualizado:")
pi_equilibrio

Delta calibrado (prêmio assumido 3.0% a.a. / variância histórica da carteira de referência): 0.29

Pesos estratégicos (w_ref, ILUSTRATIVO — substituir pela política real da GHIA):
cdi           0.555556
dolar         0.111111
prefixado     0.166667
ipca_longo    0.166667
Name: peso_estrategico, dtype: float64

Retorno de equilíbrio implícito (Π), anualizado:


cdi            0.047682
dolar          6.930417
prefixado      1.844887
ipca_longo    10.940953
Name: retorno_equilibrio_anual_pct, dtype: float64

## 3.5 Visão única do comitê, calibrada por Idzorek

Uma visão absoluta por ativo (a mesa acredita que o retorno esperado de um ativo é X,
com Y% de confiança) — escopo de v1 do briefing é uma única visão. Idzorek troca a
matriz de incerteza `Ω`, difícil de justificar para quem não é técnico, por um
percentual de confiança (0 a 100%) — 0% deixa o posterior igual ao prior; 100% força
o posterior a bater exatamente com a visão.

In [29]:
# Visão ilustrativa, para exercitar a mecânica: a mesa acredita que a bolsa vai entregar
# 4 p.p. acima do que o equilíbrio embutido sugere, com 50% de confiança. Trocar pelo que
# o comitê de fato disser no mês de referência.
ATIVO_DA_VISAO = "bolsa" if "bolsa" in ATIVOS else ATIVOS[-1]  # bolsa se disponível; senão, o último ativo da lista
PREMIO_DA_VISAO_PP = 4.0
CONFIANCA_DA_VISAO = 0.5

visao_absoluta = pd.Series({ATIVO_DA_VISAO: pi_equilibrio[ATIVO_DA_VISAO] + PREMIO_DA_VISAO_PP})

bl = BlackLittermanModel(
    cov_anual,
    pi=pi_equilibrio,
    absolute_views=visao_absoluta,
    omega="idzorek",
    view_confidences=[CONFIANCA_DA_VISAO],
    risk_aversion=DELTA_AVERSAO_RISCO,
)

retorno_posterior = bl.bl_returns()
print(f"Visão: {ATIVO_DA_VISAO} em {pi_equilibrio[ATIVO_DA_VISAO] + PREMIO_DA_VISAO_PP:.2f}% a.a. "
      f"(equilíbrio + {PREMIO_DA_VISAO_PP:.1f}p.p.), confiança {CONFIANCA_DA_VISAO*100:.0f}%.")
print("\nRetorno: equilíbrio (prior) vs. posterior (depois da visão):")
pd.DataFrame({"prior_pi": pi_equilibrio, "posterior_bl": retorno_posterior})

Visão: ipca_longo em 14.94% a.a. (equilíbrio + 4.0p.p.), confiança 50%.

Retorno: equilíbrio (prior) vs. posterior (depois da visão):


,prior_pi,posterior_bl
cdi,0.047682,0.031361
dolar,6.930417,6.963420
prefixado,1.844887,2.130241
ipca_longo,10.940953,12.940953


## 3.6 Pesos antes e depois

In [30]:
pesos_depois = bl.bl_weights(risk_aversion=DELTA_AVERSAO_RISCO)
pesos_depois = pd.Series(pesos_depois) / pd.Series(pesos_depois).sum()  # normaliza para soma 1 (comparável ao w_ref)

pesos_antes_depois = pd.DataFrame({"antes (w_ref, política)": W_REF_ILUSTRATIVO, "depois (posterior BL)": pesos_depois})
pesos_antes_depois["diferenca_pp"] = (pesos_antes_depois["depois (posterior BL)"] - pesos_antes_depois["antes (w_ref, política)"]) * 100

registrar_log("black_litterman_v1", "ok", {
    "ativos": ATIVOS, "ativo_da_visao": ATIVO_DA_VISAO, "premio_pp": PREMIO_DA_VISAO_PP,
    "confianca": CONFIANCA_DA_VISAO, "pesos_antes_depois": pesos_antes_depois.round(4).to_dict(orient="index"),
})
pesos_antes_depois.round(4)

2026-09-18 03:54:31,115 | INFO | [black_litterman_v1] ok | {'ativos': ['cdi', 'dolar', 'prefixado', 'ipca_longo'], 'ativo_da_visao': 'ipca_longo', 'premio_pp': 4.0, 'confianca': 0.5, 'pesos_antes_depois': {'cdi': {'antes (w_ref, política)': 0.5556, 'depois (posterior BL)': 0.5371, 'diferenca_pp': -1.8434}, 'dolar': {'antes (w_ref, política)': 0.1111, 'depois (posterior BL)': 0.1074, 'diferenca_pp': -0.3687}, 'prefixado': {'antes (w_ref, política)': 0.1667, 'depois (posterior BL)': 0.1611, 'diferenca_pp': -0.553}, 'ipca_longo': {'antes (w_ref, política)': 0.1667, 'depois (posterior BL)': 0.1943, 'diferenca_pp': 2.7651}}}


,"antes (w_ref, política)",depois (posterior BL),diferenca_pp
cdi,0.5556,0.5371,-1.8434
dolar,0.1111,0.1074,-0.3687
prefixado,0.1667,0.1611,-0.5530
ipca_longo,0.1667,0.1943,2.7651


## 3.7 Registro da visão (para o backtest de seis meses)

O argumento de venda não é a carteira sugerida — é comparar, meses depois, se a visão
registrada agregou ou destruiu valor. Esta função só acumula um log; o backtest em si
(comparar visão vs. realizado) roda quando já houver histórico suficiente acumulado.

In [31]:
CAMINHO_VISOES = DATA_DIR / "visoes_comite.jsonl"


def registrar_visao_comite(ativo: str, retorno_esperado_pct: float, confianca: float, justificativa: str, data_referencia=None) -> None:
    """
    Acrescenta uma visão do comitê ao log de visões (JSON Lines, um registro por
    chamada). Base do backtest de visão da Fase 3: comparar, alguns meses depois, o
    `retorno_esperado_pct` registrado aqui contra o retorno realizado no período.
    """
    registro = {
        "timestamp_registro": datetime.now().isoformat(timespec="seconds"),
        "data_referencia": str(pd.Timestamp(data_referencia or datetime.now()).date()),
        "ativo": ativo,
        "retorno_esperado_pct": retorno_esperado_pct,
        "confianca": confianca,
        "justificativa": justificativa,
    }
    with open(CAMINHO_VISOES, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")
    print(f"Visão registrada: {registro}")


registrar_visao_comite(
    ativo=ATIVO_DA_VISAO,
    retorno_esperado_pct=float(pi_equilibrio[ATIVO_DA_VISAO] + PREMIO_DA_VISAO_PP),
    confianca=CONFIANCA_DA_VISAO,
    justificativa="Visão ilustrativa de exemplo (Seção 3.5) — substituir pelo que o comitê disser de fato.",
)

Visão registrada: {'timestamp_registro': '2026-09-18T03:54:31', 'data_referencia': '2026-09-18', 'ativo': 'ipca_longo', 'retorno_esperado_pct': 14.94095292018919, 'confianca': 0.5, 'justificativa': 'Visão ilustrativa de exemplo (Seção 3.5) — substituir pelo que o comitê disser de fato.'}
